# Silver: conform the record, and keep "not asked" separate from "no"

Types the two feeds, resolves the specialty spellings, attaches the HPO label each
feature will be cited by, and assigns every feature to a body system.

| | |
| --- | --- |
| **Reads** | `bronze_*` |
| **Writes** | `silver_patients`, `silver_encounters`, `silver_observations`, `silver_family_history`, `silver_quarantine` |

## The distinction this layer exists to protect

Family history is recorded for about 60% of the cohort. The obvious move is to fill the
gap with `false` and get a clean boolean. That would be wrong, and dangerous:

* `asked = true, affected = false` means a clinician asked and the answer was no.
* `asked = false` means **nobody asked**. It says nothing about the patient.

Collapsing the second into the first invents a negative finding for 40% of the cohort,
and every downstream count of "patients with no family history" becomes a lie. Silver
keeps `history_taken` as its own column and Gold refuses to score on the absence.

The same reasoning drives the screening window: a child whose record is too short to
read is `not_screened`, which is not the same as screened-and-clear.

## Dates

Two feeds, two formats, both ambiguous for the first twelve days of any month. Parsed
with the format each feed actually writes rather than by guessing per row - a row that
will not parse goes to quarantine rather than silently becoming a wrong date.

In [ ]:
MIN_RECORD_DAYS = 180
MIN_POPULATED_RATE = 0.97
PIPELINE_RUN_ID = ""

In [ ]:
import notebookutils
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

RUN_ID = PIPELINE_RUN_ID or "local"

_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table):
    """Read a table from a lakehouse other than the attached default.

    spark.read.table() resolves only against the default lakehouse, so a cross-lakehouse
    read has to go by OneLake path -- and that path needs the lakehouse *id*. Mixing the
    workspace id with the lakehouse *name* returns a 400 from the ABFS driver, which
    surfaces as an opaque Py4JJavaError rather than anything mentioning names.
    """
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    return spark.read.format("delta").load(
        f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table}")


BRONZE = "bronze_lakehouse"
print("workspace", _WS)

In [ ]:
# ------------------------------------------------- HPO labels and body systems
# The system each feature belongs to. Features spread across several systems is the
# single strongest reason a paediatrician refers, so this mapping is load-bearing --
# it is stated here, in one place, rather than implied by a score.
BODY_SYSTEM = {
    "HP:0001263": "neurodevelopment", "HP:0001249": "neurodevelopment",
    "HP:0000750": "neurodevelopment", "HP:0002376": "neurodevelopment",
    "HP:0001252": "neurology", "HP:0001250": "neurology",
    "HP:0004322": "growth", "HP:0001518": "growth", "HP:0011968": "growth",
    "HP:0000252": "craniofacial", "HP:0000175": "craniofacial",
    "HP:0001999": "craniofacial",
    "HP:0001627": "cardiac",
    "HP:0000365": "sensory", "HP:0000505": "sensory",
    "HP:0002650": "skeletal",
}

system_rows = [{"hpo_id": k, "body_system": v} for k, v in BODY_SYSTEM.items()]
system_schema = StructType([StructField("hpo_id", StringType()),
                            StructField("body_system", StringType())])
systems = spark.createDataFrame(system_rows, system_schema)

hpo = (lake_table(BRONZE, "bronze_hpo_terms")
       .select("hpo_id", F.col("name").alias("hpo_label"),
               F.col("definition").alias("hpo_definition")))

terms = hpo.join(systems, "hpo_id", "full_outer")

# A feature we cannot label cannot be cited, and a feature with no system cannot be
# counted toward multi-system involvement. Both are build-stopping.
unlabelled = terms.filter(F.col("hpo_label").isNull()).count()
unmapped = terms.filter(F.col("body_system").isNull()).count()
print(f"terms {terms.count()}  unlabelled={unlabelled}  unmapped={unmapped}")
if unlabelled or unmapped:
    terms.filter(F.col("hpo_label").isNull() | F.col("body_system").isNull()).show(
        truncate=False)
    raise ValueError(
        f"{unlabelled} terms without a label and {unmapped} without a body system. "
        f"The evidence contract cites labels and counts systems, so neither can be "
        f"left to a downstream default.")

terms.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("silver_hpo_terms")
print("wrote silver_hpo_terms")

In [ ]:
# --------------------------------------------------------------- patients
# _latent_cluster is deliberately dropped here. It is the generator's answer key, and
# nothing downstream is allowed to see it -- that is what makes the pipeline's output
# a screening result rather than a lookup.
patients = (lake_table(BRONZE, "bronze_patients")
            .withColumn("record_days", F.datediff(F.lit("2026-08-27").cast("date"),
                                                  F.col("enrolled_on")))
            .withColumn("age_years",
                        F.floor(F.datediff(F.lit("2026-08-27").cast("date"),
                                           F.col("birth_date")) / 365.25)
                        .cast("int"))
            .withColumn("screenable", F.col("record_days") >= MIN_RECORD_DAYS)
            .drop("_latent_cluster"))

assert "_latent_cluster" not in patients.columns, "answer key leaked into silver"

patients.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("silver_patients")

total = patients.count()
screenable = patients.filter("screenable").count()
print(f"patients          {total:>6,}")
print(f"screenable        {screenable:>6,}  ({screenable / total:.1%})")
print(f"too little record {total - screenable:>6,}  -> not_screened, NOT a clear screen")

In [ ]:
# ------------------------------------------------------------- encounters
# The encounter feed writes dd/MM/yyyy. Parse with that format; anything that will not
# parse is quarantined rather than coerced.
raw_encounters = lake_table(BRONZE, "bronze_encounters")

specialty = F.initcap(F.trim(F.col("specialty_raw")))
# "Pediatrics" and "Paediatrics" are the same service; so are the bracketed variants.
specialty = F.regexp_replace(specialty, "Pediatric", "Paediatric")
specialty = F.regexp_replace(specialty, r"\s*\(.*\)$", "")
specialty = F.regexp_replace(specialty, "^Otolaryngology$", "Ent")
specialty = F.regexp_replace(specialty, "Orthopedic", "Orthopaedic")

encounters = (raw_encounters
              .withColumn("encounter_date",
                          F.to_date("encounter_date_raw", "dd/MM/yyyy"))
              .withColumn("specialty", specialty))

bad_dates = encounters.filter(F.col("encounter_date").isNull())
print(f"encounters with an unparseable date: {bad_dates.count()}")

(bad_dates.select("encounter_id", F.lit("encounter").alias("source_table"),
                  F.col("encounter_date_raw").alias("offending_value"),
                  F.lit("date did not parse as dd/MM/yyyy").alias("reason"))
 .write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable("silver_quarantine"))

encounters = encounters.filter(F.col("encounter_date").isNotNull()).drop(
    "encounter_date_raw", "specialty_raw")
encounters.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("silver_encounters")

print(f"encounters        {encounters.count():>6,}")
print("\ndistinct specialties after conforming:")
for row in (encounters.groupBy("specialty").count()
            .orderBy(F.desc("count")).collect()):
    print(f"  {row['specialty']:28} {row['count']:>6,}")

In [ ]:
# ----------------------------------------------------------- observations
# The observation feed writes MM/dd/yyyy -- the other way round.
raw_observations = lake_table(BRONZE, "bronze_observations")

observations = (raw_observations
                .withColumn("observed_date",
                            F.to_date("observed_date_raw", "MM/dd/yyyy"))
                .withColumn("recorded_by",
                            F.initcap(F.trim(F.col("recorded_by_raw"))))
                .withColumn("recorded_by",
                            F.regexp_replace(F.col("recorded_by"), "^Np$",
                                             "Nurse Practitioner")))

bad_obs = observations.filter(F.col("observed_date").isNull()).count()
print(f"observations with an unparseable date: {bad_obs}")

observations = (observations.filter(F.col("observed_date").isNotNull())
                .join(spark.table("silver_hpo_terms").select(
                    "hpo_id", "hpo_label", "body_system"), "hpo_id", "left")
                .drop("observed_date_raw", "recorded_by_raw"))

orphan_terms = observations.filter(F.col("hpo_label").isNull()).count()
if orphan_terms:
    raise ValueError(
        f"{orphan_terms} observations reference an HPO term with no label. Every "
        f"observation becomes a citable piece of evidence, so an unlabelled one would "
        f"be cited as a bare code the clinician cannot look up.")

observations.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("silver_observations")
print(f"observations      {observations.count():>6,}")

In [ ]:
# ---------------------------------------------------------- family history
# The whole point of this cell: every patient gets a row, and the row records whether
# the question was ever asked. A left join alone would leave nulls that the next
# person to read this would very reasonably read as "no".
history = (lake_table(BRONZE, "bronze_family_history")
           .withColumn("asked_on", F.to_date("asked_on_raw", "dd/MM/yyyy"))
           .drop("asked_on_raw", "run_id"))

family = (spark.table("silver_patients").select("patient_id")
          .join(history, "patient_id", "left")
          .withColumn("history_taken", F.col("asked_on").isNotNull()))

# Where the history was never taken, the three flags are NULL -- unknown -- not false.
for flag in ["affected_first_degree", "consanguinity", "recurrent_pregnancy_loss"]:
    family = family.withColumn(
        flag, F.when(F.col("history_taken"), F.col(flag)).otherwise(F.lit(None)))

family.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("silver_family_history")

taken = family.filter("history_taken").count()
total = family.count()
positive = family.filter(F.col("affected_first_degree") == True).count()
print(f"patients                       {total:>6,}")
print(f"family history taken           {taken:>6,}  ({taken / total:.0%})")
print(f"  of those, affected relative  {positive:>6,}")
print(f"never asked                    {total - taken:>6,}  <- unknown, not negative")

In [ ]:
# ------------------------------------------------------------------ gates
# A populated-rate gate on the columns the evidence contract will cite. A column that
# is quietly 30% null produces an agent that quietly omits a third of the evidence.
CRITICAL = {
    "silver_patients": ["patient_id", "birth_date", "enrolled_on", "screenable"],
    "silver_encounters": ["encounter_id", "patient_id", "encounter_date", "specialty"],
    "silver_observations": ["observation_id", "patient_id", "observed_date",
                            "hpo_id", "hpo_label", "body_system"],
    "silver_family_history": ["patient_id", "history_taken"],
}

failures = []
for table, columns in CRITICAL.items():
    frame = spark.table(table)
    rows = frame.count()
    for column in columns:
        populated = frame.filter(F.col(column).isNotNull()).count()
        rate = populated / rows if rows else 0.0
        flag = "ok " if rate >= MIN_POPULATED_RATE else "FAIL"
        print(f"  {flag} {table}.{column:22} {rate:6.1%}")
        if rate < MIN_POPULATED_RATE:
            failures.append(f"{table}.{column} at {rate:.1%}")

if failures:
    raise ValueError("populated-rate gate failed: " + "; ".join(failures))
print("\nsilver complete")